# Customer Churn Prediction
**CodSoft ML Internship** — Random Forest Classifier

> Replace the dummy dataset with the Telco Customer Churn CSV from Kaggle.
> ```python
> df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
> ```

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)
plt.rcParams.update({'figure.dpi':110})
print('✅ Imports done')

## 📂 Load Dataset

In [ ]:
# ── Replace with your real CSV ──────────────────────────────
# df = pd.read_csv(r'D:\codsoft\WA_Fn-UseC_-Telco-Customer-Churn.csv')
# df['Churn'] = df['Churn'].map({'Yes':1,'No':0})

def make_churn_data(n=5000, churn_rate=0.27, seed=42):
    rng = np.random.default_rng(seed)
    nc = int(n*churn_rate); ns = n-nc
    def block(size, churn):
        return pd.DataFrame({
            'tenure': rng.integers(1,72,size),
            'MonthlyCharges': rng.uniform(20,120,size),
            'TotalCharges': rng.uniform(20,8000,size),
            'SeniorCitizen': rng.integers(0,2,size),
            'gender': rng.choice(['Male','Female'],size),
            'Partner': rng.choice(['Yes','No'],size),
            'Dependents': rng.choice(['Yes','No'],size),
            'PhoneService': rng.choice(['Yes','No'],size),
            'MultipleLines': rng.choice(['Yes','No','No phone service'],size),
            'InternetService': rng.choice(['DSL','Fiber optic','No'],size),
            'OnlineSecurity': rng.choice(['Yes','No','No internet service'],size),
            'TechSupport': rng.choice(['Yes','No','No internet service'],size),
            'Contract': rng.choice(['Month-to-month','One year','Two year'],size),
            'PaperlessBilling': rng.choice(['Yes','No'],size),
            'PaymentMethod': rng.choice(['Electronic check','Mailed check','Bank transfer','Credit card'],size),
            'Churn': [churn]*size
        })
    return pd.concat([block(ns,0),block(nc,1)]).sample(frac=1,random_state=seed).reset_index(drop=True)

df = make_churn_data()
print(f'Shape: {df.shape} | Churn rate: {df["Churn"].mean()*100:.1f}%')
df.head()

## 🔍 EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Churn distribution
vc = df['Churn'].value_counts()
axes[0,0].pie(vc.values, labels=['No Churn','Churn'], autopct='%1.1f%%',
              colors=['#3498db','#e74c3c'], startangle=90)
axes[0,0].set_title('Churn Distribution', fontweight='bold')

# Monthly charges by churn
for val, col, lab in [(0,'#3498db','No Churn'),(1,'#e74c3c','Churn')]:
    axes[0,1].hist(df[df['Churn']==val]['MonthlyCharges'], bins=30, alpha=0.7, color=col, label=lab)
axes[0,1].set_title('Monthly Charges', fontweight='bold'); axes[0,1].legend()

# Tenure by churn
df.boxplot(column='tenure', by='Churn', ax=axes[0,2], patch_artist=True)
axes[0,2].set_title('Tenure by Churn', fontweight='bold'); plt.sca(axes[0,2]); plt.title('Tenure by Churn')

# Contract type
ct = df.groupby(['Contract','Churn']).size().unstack(fill_value=0)
ct.plot(kind='bar', ax=axes[1,0], color=['#3498db','#e74c3c'], edgecolor='black')
axes[1,0].set_title('Contract Type vs Churn', fontweight='bold')
axes[1,0].set_xlabel(''); plt.setp(axes[1,0].xaxis.get_majorticklabels(), rotation=25, ha='right')

# Internet service
is_ = df.groupby(['InternetService','Churn']).size().unstack(fill_value=0)
is_.plot(kind='bar', ax=axes[1,1], color=['#3498db','#e74c3c'], edgecolor='black')
axes[1,1].set_title('Internet Service vs Churn', fontweight='bold')
axes[1,1].set_xlabel(''); plt.setp(axes[1,1].xaxis.get_majorticklabels(), rotation=25, ha='right')

# Total charges distribution
for val, col, lab in [(0,'#3498db','No Churn'),(1,'#e74c3c','Churn')]:
    axes[1,2].hist(df[df['Churn']==val]['TotalCharges'], bins=30, alpha=0.7, color=col, label=lab)
axes[1,2].set_title('Total Charges', fontweight='bold'); axes[1,2].legend()

plt.suptitle('Customer Churn — EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight', dpi=120)
plt.show()

## 🤖 Preprocessing & Training

In [ ]:
feature_cols = [c for c in df.columns if c != 'Churn']
df_enc = df.copy()

num_cols = ['tenure','MonthlyCharges','TotalCharges']
cat_cols = [c for c in feature_cols if c not in num_cols]

for c in cat_cols:
    df_enc[c] = LabelEncoder().fit_transform(df_enc[c].astype(str))

df_enc[num_cols] = SimpleImputer(strategy='median').fit_transform(df_enc[num_cols])
df_enc[num_cols] = StandardScaler().fit_transform(df_enc[num_cols])

X = df_enc[feature_cols].values
y = df_enc['Churn'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                               max_depth=15, min_samples_leaf=2,
                               random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%\n')
print(classification_report(y_test, y_pred, target_names=['No Churn','Churn'], zero_division=0))
print('✅ Training complete')

## 📈 Evaluation

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={auc:.3f}')
axes[1].plot([0,1],[0,1],'--',color='gray'); axes[1].legend()
axes[1].set(xlabel='FPR', ylabel='TPR'); axes[1].set_title('ROC Curve', fontweight='bold')

plt.suptitle('Customer Churn — Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Feature importance
fi = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=fi.head(15), x='importance', y='feature', palette='mako', ax=ax)
ax.set_title('Top-15 Feature Importances', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=120)
plt.show()

print('Top 5 churn drivers:')
print(fi.head().to_string(index=False))